<a href="https://colab.research.google.com/github/mithun30052001/iit-aiml-assignments/blob/main/sql-intermediate-querying/Sql_Intermediate_Querying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Loading database
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


830

In [7]:
orders_query = "Select * from orders"
orders_df = pd.read_sql(orders_query, conn)
customers_query = "Select * from customers"
customers_df = pd.read_sql(customers_query, conn)
print(orders_df.head())
print("==============================")
print(customers_df.head())

   orderID customerID  employeeID                orderDate  \
0    10248      VINET           5  1996-07-04 00:00:00.000   
1    10249      TOMSP           6  1996-07-05 00:00:00.000   
2    10250      HANAR           4  1996-07-08 00:00:00.000   
3    10251      VICTE           3  1996-07-08 00:00:00.000   
4    10252      SUPRD           4  1996-07-09 00:00:00.000   

              requiredDate              shippedDate  shipVia  freight  \
0  1996-08-01 00:00:00.000  1996-07-16 00:00:00.000        3    32.38   
1  1996-08-16 00:00:00.000  1996-07-10 00:00:00.000        1    11.61   
2  1996-08-05 00:00:00.000  1996-07-12 00:00:00.000        2    65.83   
3  1996-08-05 00:00:00.000  1996-07-15 00:00:00.000        1    41.34   
4  1996-08-06 00:00:00.000  1996-07-11 00:00:00.000        2    51.30   

                    shipName          shipAddress        shipCity shipRegion  \
0  Vins et alcools Chevalier   59 rue de l'Abbaye           Reims       None   
1         Toms Spezialitäten

In [11]:
#Task 1 — Aggregation and Grouping
agg_and_group_query = """Select customerID, count(orderID) as order_count,
                          sum(freight) as total_freight, avg(freight) as avg_freight
                          from orders
                          group by customerID order by total_freight desc"""
agg_and_group_result = pd.read_sql(agg_and_group_query, conn)
print(f"Top 10 rows:\n {agg_and_group_result.head(10)}")

Top 10 rows:
   customerID  order_count  total_freight  avg_freight
0      SAVEA           31        6683.70   215.603226
1      ERNSH           30        6205.39   206.846333
2      QUICK           28        5605.63   200.201071
3      HUNGO           19        2755.24   145.012632
4      RATTC           18        2134.21   118.567222
5      QUEEN           13        1982.70   152.515385
6      FOLKO           19        1678.08    88.320000
7      BERGS           18        1559.52    86.640000
8      FRANK           15        1403.44    93.562667
9      MEREP           13        1394.22   107.247692


In [13]:
#Task 2 — WHERE vs. HAVING
#Query A — Using WHERE
filter_before_agg_query = """SELECT customerID,
                          COUNT(orderID) AS high_freight_orders
                          FROM orders
                          WHERE freight > 50
                          GROUP BY customerID;"""
filter_before_agg_result = pd.read_sql(filter_before_agg_query, conn)
print(f"Using Where:\n {filter_before_agg_result}")

Using Where:
    customerID  high_freight_orders
0       ALFKI                    2
1       ANTON                    2
2       AROUT                    2
3       BERGS                   11
4       BLAUS                    1
..        ...                  ...
69      WANDK                    2
70      WARTH                    6
71      WELLI                    1
72      WHITC                    7
73      WOLZA                    1

[74 rows x 2 columns]


In [15]:
#Query B — Using HAVING
filter_after_agg_query = """SELECT customerID,
                            SUM(freight) AS total_freight
                            FROM orders
                            GROUP BY customerID
                            HAVING SUM(freight) > 500;"""
filter_after_agg_result = pd.read_sql(filter_after_agg_query, conn)
print(f"Using Having:\n {filter_after_agg_result}")

Using Having:
    customerID  total_freight
0       BERGS        1559.52
1       BLONP         623.66
2       BONAP        1357.87
3       BOTTM         793.95
4       EASTC         832.34
5       ERNSH        6205.39
6       FOLIG         637.94
7       FOLKO        1678.08
8       FRANK        1403.44
9       GODOS         568.27
10      GREAL        1087.61
11      HANAR         724.77
12      HILAA        1259.16
13      HUNGO        2755.24
14      KOENE         813.68
15      LAMAI         635.82
16      LEHMS        1017.03
17      LILAS         734.41
18      LINOD         673.81
19      MEREP        1394.22
20      OLDWO         983.53
21      OTTIK         862.74
22      PICCO        1186.11
23      QUEEN        1982.70
24      QUICK        5605.63
25      RATTC        2134.21
26      RICAR         632.95
27      RICSU        1001.29
28      SAVEA        6683.70
29      SEVES         913.81
30      SPLIR         558.67
31      SUPRD         821.23
32      VAFFE         947.34

In [16]:
''' Explanation
WHERE filters individual rows before grouping happens, so in Query A only orders with freight greater than 50 are considered when counting.

HAVING filters after aggregation, so in Query B all orders are first grouped by customer and their total freight is calculated, and only then customers exceeding the total freight threshold are returned.'''

' Explanation\nWHERE filters individual rows before grouping happens, so in Query A only orders with freight greater than 50 are considered when counting.\n\nHAVING filters after aggregation, so in Query B all orders are first grouped by customer and their total freight is calculated, and only then customers exceeding the total freight threshold are returned.'

In [18]:
#Task 3 — JOIN and Aggregation
#Query1 : only customers with orders
customers_with_orders = '''SELECT c.CompanyName,
                          c.Country,
                          COUNT(o.orderID) AS order_count,
                          SUM(o.freight) AS total_freight
                          FROM customers c
                          INNER JOIN orders o
                          ON c.customerID = o.customerID
                          GROUP BY c.CompanyName, c.Country;'''
customers_with_orders_result = pd.read_sql(customers_with_orders, conn)
print(f"Customers with orders:\n {customers_with_orders_result}")

Customers with orders:
                            companyName  country  order_count  total_freight
0                  Alfreds Futterkiste  Germany            6         225.58
1   Ana Trujillo Emparedados y helados   Mexico            4          97.42
2              Antonio Moreno Taquería   Mexico            7         268.52
3                      Around the Horn       UK           13         471.95
4                        B's Beverages       UK           10         281.31
..                                 ...      ...          ...            ...
84                      Wartian Herkku  Finland           15         822.48
85              Wellington Importadora   Brazil            9         194.71
86                White Clover Markets      USA           14        1353.06
87                         Wilman Kala  Finland            7          88.41
88                      Wolski  Zajazd   Poland            7         175.74

[89 rows x 4 columns]


In [20]:
#Query2 : all customers, including no orders
all_customers = '''SELECT c.CompanyName,
                  c.Country,
                  COUNT(o.orderID) AS order_count,
                  COALESCE(SUM(o.freight), 0) AS total_freight
                  FROM customers c
                  LEFT JOIN orders o
                  ON c.customerID = o.customerID
                  GROUP BY c.CompanyName, c.Country;'''
all_customers_result = pd.read_sql(all_customers, conn)
print(f"All customers:\n {all_customers_result}")

All customers:
                            companyName  country  order_count  total_freight
0                  Alfreds Futterkiste  Germany            6         225.58
1   Ana Trujillo Emparedados y helados   Mexico            4          97.42
2              Antonio Moreno Taquería   Mexico            7         268.52
3                      Around the Horn       UK           13         471.95
4                        B's Beverages       UK           10         281.31
..                                 ...      ...          ...            ...
86                      Wartian Herkku  Finland           15         822.48
87              Wellington Importadora   Brazil            9         194.71
88                White Clover Markets      USA           14        1353.06
89                         Wilman Kala  Finland            7          88.41
90                      Wolski  Zajazd   Poland            7         175.74

[91 rows x 4 columns]


In [21]:
'''Explanation
INNER JOIN returns only customers who have matching rows in the orders table, so customers without any orders are excluded entirely.

LEFT JOIN includes all customers, even if they have no matching orders; in such cases, the order-related columns become NULL. Using COALESCE ensures that total freight appears as 0 instead of NULL for those customers.

The key difference is that INNER JOIN filters out non-matching rows, while LEFT JOIN preserves them.'''

'Explanation\nINNER JOIN returns only customers who have matching rows in the orders table, so customers without any orders are excluded entirely.\n\nLEFT JOIN includes all customers, even if they have no matching orders; in such cases, the order-related columns become NULL. Using COALESCE ensures that total freight appears as 0 instead of NULL for those customers.\n\nThe key difference is that INNER JOIN filters out non-matching rows, while LEFT JOIN preserves them.'